<a href="https://colab.research.google.com/github/tranmailinheecneu-arch/bank-failure-pipeline/blob/main/part5_mllib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, glob, subprocess
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True)
sh("apt-get update -qq && apt-get install -y openjdk-17-jdk-headless")
sh("pip -q install pyspark==4.0.1 pyarrow")
cand = sorted(glob.glob("/usr/lib/jvm/*17*"))
os.environ["JAVA_HOME"] = cand[0]
os.environ["PATH"] = cand[0] + "/bin:" + os.environ["PATH"]
print(sh("java -version").stderr)

openjdk version "17.0.21.1" 2026-11-17
OpenJDK Runtime Environment (build 17.0.21.1+2-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.1+2-1-24.04.1-Ubuntu, mixed mode, sharing)



In [ ]:
import os, sys, time
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.master("local[*]").config("spark.driver.memory", "5g")
         .config("spark.sql.shuffle.partitions", "16").getOrCreate())
log = []
def say(s=""):
    log.append(str(s)); print(s)

Q = lambda y, qt: y * 4 + qt - 1
df = spark.read.parquet("/content/mllib_input.parquet")
FEATURES = [c for c in df.columns if c not in ("CERT", "qidx", "y", "y_before_public")]
df = df.select("qidx", "y", "y_before_public",
               *[F.when(F.isnan(F.col(c)), None).otherwise(F.col(c).cast("double")).alias(c) for c in FEATURES])
tr = df.filter(F.col("qidx") <= Q(1994, 4))                                   # fold 4 training rows
va = df.filter((F.col("qidx") >= Q(1996, 1)) & (F.col("qidx") <= Q(2006, 4))
               & ~((F.col("y") == 1) & (F.col("y_before_public") == 1)))      # fold 4 validation, actionable rows

# clip bounds learned on the TRAINING rows only (same rule as the scikit-learn QuantileClipper)
bounds = tr.approxQuantile(FEATURES, [0.005, 0.995], 0.001)
def clip(d):
    return d.select("y", *[F.when(F.col(c).isNull(), None).otherwise(
        F.least(F.greatest(F.col(c), F.lit(b[0])), F.lit(b[1]))).alias(c) for c, b in zip(FEATURES, bounds)])
tr, va = clip(tr).cache(), clip(va).cache()
n_pos = tr.filter("y = 1").count(); n_neg = tr.count() - n_pos
tr = tr.withColumn("w", F.when(F.col("y") == 1, n_neg / n_pos).otherwise(1.0))   # balanced class weight
nv, nvp = va.count(), va.filter("y = 1").count()
say(f"train rows {n_pos + n_neg:,} (positive {n_pos:,}); validation rows {nv:,} (positive {nvp:,}); validation base rate {nvp / nv:.4%}")

imp = Imputer(inputCols=FEATURES, outputCols=[c + "_i" for c in FEATURES], strategy="median")
asm = VectorAssembler(inputCols=[c + "_i" for c in FEATURES], outputCol="raw", handleInvalid="keep")
sc = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True)
ev_pr = BinaryClassificationEvaluator(labelCol="y", metricName="areaUnderPR")
ev_roc = BinaryClassificationEvaluator(labelCol="y", metricName="areaUnderROC")
for name, clf in [("logistic regression (weighted)", LogisticRegression(labelCol="y", featuresCol="features", weightCol="w", maxIter=100, regParam=1e-4)),
                  ("gradient-boosted trees (30 iterations, depth 4)", GBTClassifier(labelCol="y", featuresCol="raw", weightCol="w", maxIter=30, maxDepth=4, seed=42))]:
    stages = [imp, asm] + ([sc] if "logistic" in name else []) + [clf]
    t = time.time(); model = Pipeline(stages=stages).fit(tr); pred = model.transform(va)
    say(f"{name}: area under PR {ev_pr.evaluate(pred):.4f} | area under ROC {ev_roc.evaluate(pred):.4f} | {time.time() - t:.0f} s")
say("Spark's areaUnderPR uses trapezoidal interpolation: close to, but not identical with, scikit-learn's Average Precision.")
say("Compare with the fold-4 results of the same model families in results/part5_validation.txt (AP_f4, pooled_AUC).")
open("/content/part5_mllib_results.txt", "w").write("\n".join(log))
files.download("/content/part5_mllib_results.txt")

train rows 704,362 (positive 8,488); validation rows 440,235 (positive 163); validation base rate 0.0370%
logistic regression (weighted): area under PR 0.0400 | area under ROC 0.8810 | 90 s
gradient-boosted trees (30 iterations, depth 4): area under PR 0.0826 | area under ROC 0.8963 | 116 s
Spark's areaUnderPR uses trapezoidal interpolation: close to, but not identical with, scikit-learn's Average Precision.
Compare with the fold-4 results of the same model families in results/part5_validation.txt (AP_f4, pooled_AUC).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>